# C0 · synthetic round-width arm: does a narrower *round* blob help as much as a narrow oval axis? (exploratory)

**Why.** In the synthetic mechanism study (72 runs) narrowing one axis of an equal-area oval reduced the error along that axis
(a_narrow 0.07–0.65), unlike on ISBI. But an oval narrows one axis *and* widens the other, and the study had no round-width arm,
so it cannot say whether a plain narrower round blob gives the same gain. That is the fair comparator for MDT: an oval is only worth
its complexity if it beats simply shrinking the round blob.

**Design.** Same images, test set, batch stream, network, loss, decoders and `SYNTH_PROTOCOL` as the mechanism study (the hash is
asserted). Round targets of width w × σ₀ with w = 0.5, 0.71, 1.41, 2 (σ₀ = 4 px → 2, 2.83, 5.66, 8 px). The 2 px round blob has
the width of the ratio-4 oval's short axis, the 8 px blob that of its long axis. N = 1,600 and 6,400 (N = 100 and 400 were unusable
or marginal in the mechanism study); tiers Extra-low, Low, High; seeds 1, 2. 4 × 2 × 3 × 2 = **48 runs**. The round runs pair with
the mechanism study's B0 (w = 1), A∥ and A⊥ runs of the same tier, N and seed, which must be present in `RUN_ROOT` for the analysis.

**Questions (separate analysis notebook).** (1) Round size response per direction: Δ log SD vs log width, separately for narrowing and widening.
(2) Round vs oval along the same axis at the same width: oval short axis (2 px) vs round 2 px, oval long axis (8 px) vs round 8 px.
(3) Can each tier learn the narrow round blobs at all (failure rates by width)? (4) Which target has the smallest overall error?

**Status.** Exploratory, designed after the mechanism study's results; not pre-registered. Run ids `C0_SYNTH_<tier>_N<n>_R<w>_s<seed>`;
the arm spec has its own hash (`RW_HASH`), written to `config.json` and as line 3 of `DONE`.

**Running it on a pod.** Six GPU copies (`_gpu0_1` … `_gpu1_3`), 8 runs each, started together with the same `RUN_ROOT`. These
notebooks only train; nothing from the mechanism study is needed on the pod. Expected: about 27 min per High run and 7 min per
Low/Extra-low run with six notebooks, i.e. roughly 2–2.5 h wall time. §9 prints a status table (runs done, MRE and failures per run).
The comparison with the mechanism study's B0/A∥/A⊥ runs is a separate notebook, `C0_SYNTH_round_width_analysis.ipynb`, run
locally after downloading the round-width runs into the same folder as the mechanism runs (`weights/C0/SYNTH`).

In [ ]:
import os

RUN_IDS = "all"          # "all" or a list of run ids from the run table (§2)
SMOKE_TEST = False       # True: 64 px images, tiny networks, 40 iterations; checks the pipeline only
GPU = None               # physical GPU index for this notebook (e.g. 0); None = first visible GPU
RUN_ROOT = os.environ.get("C0_SYNTH_ROOT", "/workspace/C0/SYNTH" if os.path.isdir("/workspace") else "/tf/weights/C0/SYNTH")
ITERS = 30000            # optimisation steps per run, the same for every N (ISBI runs: about 29,600)
BATCH_SIZE = 8
PROGRESS_BAR = True      # one progress bar per run (loss, it/s, ETA)
SAVE_MODELS = True       # final weights per run (High ~120 MB); set False to keep only predictions

## 0. Setup and frozen protocol

In [ ]:
import sys, json, math, time, hashlib, platform
from pathlib import Path
if GPU is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else
             torch.float16 if DEVICE.type == "cuda" else None)
torch.backends.cudnn.benchmark = True

IMG = 64 if SMOKE_TEST else 256
H_C = W_C = IMG
K = 2
LM_NAMES = ["corner", "edge point"]
SIGMA0_PX = 1.5 if SMOKE_TEST else 4.0
RATIO = 4.0
N_TRAIN = [20, 40] if SMOKE_TEST else [100, 400, 1600, 6400]
N_TEST = 40 if SMOKE_TEST else 1000
TIER_BASE = {"Extra-low": 4, "High": 8} if SMOKE_TEST else {"Extra-low": 8, "Low": 16, "High": 64}
SEEDS = [1, 2]
ARMS = ["B0", "Apar", "Aperp"]
if SMOKE_TEST:
    ITERS = 40
    RUN_ROOT = str(Path("c0_synth_smoke").resolve())
R_WIN = 4 if SMOKE_TEST else 15
GROSS_PX = 1e9 if SMOKE_TEST else 10.0
GEN = {"texture_amp": 0.15, "texture_sigma_frac": 3 / 256, "pixel_noise": 0.05, "blur_frac": 1 / 256,
       "wedge_angle_deg": [60.0, 120.0], "wedge_len_frac": 0.2, "bump_height_frac": 0.04, "bump_width_frac": 0.06,
       "edge_disk_radius_frac": 0.28, "edge_disk_offset_frac": 0.5, "contrast": [0.4, 0.8], "min_landmark_distance_frac": 0.35, "margin_frac": 0.22,
       "train_stream": 1, "test_stream": 2}
SYNTH_PROTOCOL = {
    "study": "C0 synthetic mechanism study (plan section 9), exploratory", "image_px": IMG, "landmarks": LM_NAMES, "generator": GEN,
    "reference_axes": {"corner": "wedge bisector", "edge point": "edge tangent at the apex"},
    "sigma0_px": SIGMA0_PX, "ratio": RATIO, "arms": {"B0": "round sigma0^2 I", "Apar": "long axis along the reference axis",
                                                     "Aperp": "long axis across the reference axis"},
    "n_train": N_TRAIN, "n_test": N_TEST, "tiers": TIER_BASE, "seeds": SEEDS,
    "target_loss_net_decoders": "verbatim from the C0 arm A source (make_targets, mass_normalised_mse, UNet 4 levels, decoders)",
    "optimiser": {"name": "AdamW", "lr": 1e-3, "weight_decay": 1e-4, "warmup_iters": max(1, ITERS // 20),
                  "schedule": "cosine to 1e-6 per iteration"},
    "iters": ITERS, "batch_size": BATCH_SIZE, "batch_stream": "default_rng([seed, N, iteration]) -> indices in range(N)",
    "augmentation": None, "convergence_check": "MRE (quadratic) on the first 50 training images every ITERS/10 iterations",
    "intensity": "per-image z-score", "amp": "bf16 autocast if supported, else fp16 with GradScaler",
    "decoders": ["argmax", "quadratic", "softargmax_local", "gaussfit"], "primary_decoder": "quadratic",
    "decoder_window_radius_px": R_WIN, "gross_failure_px": GROSS_PX, "smoke_test": SMOKE_TEST,
}
SYNTH_HASH = hashlib.sha256(json.dumps(SYNTH_PROTOCOL, sort_keys=True).encode()).hexdigest()[:16]
Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)
ENV_INFO = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None, "platform": platform.platform()}
print("synthetic protocol", SYNTH_HASH, "| device", DEVICE, "| amp", AMP_DTYPE, "| smoke", SMOKE_TEST, "| RUN_ROOT", RUN_ROOT)

# round-width arm: not part of SYNTH_PROTOCOL (which stays identical to the mechanism study), but hashed and recorded separately
EXPECTED_SYNTH_HASH = "b1804b3e5ef74a65"   # the mechanism study's protocol at the pod settings
assert SMOKE_TEST or SYNTH_HASH == EXPECTED_SYNTH_HASH, f"SYNTH_PROTOCOL differs from the mechanism study: {SYNTH_HASH}"
RW_FACTORS = [2 ** -1, 2 ** -0.5, 2 ** 0.5, 2 ** 1]          # width multiples of sigma0 (log-spaced; 0.5 and 2 = oval short / long axis)
RW_FACTOR = {f"R{f:.2f}": f for f in RW_FACTORS}              # arm name -> factor (R0.50, R0.71, R1.41, R2.00)
RW_N = [20, 40] if SMOKE_TEST else [1600, 6400]
RW_SPEC = {"arm": "round-width (exploratory)", "target": "round (w * sigma0)^2 I, unit peak, as B0", "width_factors": RW_FACTORS,
           "arm_names": list(RW_FACTOR), "n_train": RW_N, "tiers": TIER_BASE, "seeds": SEEDS, "paired_with": "B0, Apar, Aperp of the "
           "mechanism study (same tier, N, seed)", "synth_protocol_hash": SYNTH_HASH}
RW_HASH = hashlib.sha256(json.dumps(RW_SPEC, sort_keys=True).encode()).hexdigest()[:16]
print("round-width arm", RW_HASH, "| widths (px):", [round(f * SIGMA0_PX, 2) for f in RW_FACTORS], "| N:", RW_N)

## 1. C0 targets, loss, network and decoders (verbatim from the arm A source)

In [ ]:
_YY = {}
def make_targets(pts, cov, H=None, W=None):
    """pts (B,K,2) canvas px [x,y]; cov (B,K,2,2). Returns (B,K,H,W) float32 Gaussians with peak 1."""
    H, W = H or H_C, W or W_C
    key = (H, W, str(pts.device))
    if key not in _YY:
        _YY[key] = (torch.arange(W, device=pts.device, dtype=torch.float32).view(1, 1, 1, W),
                    torch.arange(H, device=pts.device, dtype=torch.float32).view(1, 1, H, 1))
    xs, ys = _YY[key]
    inv = torch.linalg.inv(cov.float())
    dx = xs - pts[..., 0, None, None]
    dy = ys - pts[..., 1, None, None]
    q = inv[..., 0, 0, None, None] * dx * dx + 2 * inv[..., 0, 1, None, None] * dx * dy + inv[..., 1, 1, None, None] * dy * dy
    return torch.exp(-0.5 * q)

def in_canvas(pts, H=None, W=None):
    """(B,K) bool: landmark lies inside the canvas (after augmentation)."""
    H, W = H or H_C, W or W_C
    return (pts[..., 0] >= 0) & (pts[..., 0] <= W - 1) & (pts[..., 1] >= 0) & (pts[..., 1] <= H - 1)

def mass_normalised_mse(pred, target, cov, valid=None):
    """Per-channel SSE divided by the analytic target mass 2*pi*sqrt(det S), averaged over valid channels.
    The analytic mass does not shrink when a target is cut by the canvas border, and landmarks pushed off the
    canvas by augmentation (valid=False) are left out of that step's loss instead of dividing by a near-zero mass."""
    mass = 2 * math.pi * torch.sqrt(torch.linalg.det(cov.float()))
    se = ((pred.float() - target) ** 2).sum(dim=(2, 3))
    per = se / mass
    if valid is None:
        return per.mean()
    valid = valid.to(per.dtype)
    return (per * valid).sum() / valid.sum().clamp_min(1)

class DoubleConv(nn.Sequential):
    def __init__(self, ci, co):
        super().__init__(nn.Conv2d(ci, co, 3, padding=1, bias=True), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, padding=1, bias=True), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=K, base=64, levels=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(levels + 1)]
        self.down = nn.ModuleList()
        c = in_ch
        for i in range(levels):
            self.down.append(DoubleConv(c, ch[i])); c = ch[i]
        self.bottom = DoubleConv(ch[levels - 1], ch[levels])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(levels))])
        self.dec = nn.ModuleList([DoubleConv(2 * ch[i], ch[i]) for i in reversed(range(levels))])
        self.head = nn.Conv2d(ch[0], out_ch, 1)

    def forward(self, x):
        skips = []
        for blk in self.down:
            x = blk(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.bottom(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)

def heatmap_windows(hm, r=None):
    """hm (B,K,H,W) float tensor -> integer peaks (B,K,2) [x,y] and windows (B,K,2r+1,2r+1) as numpy (NaN outside image)."""
    r = r or R_WIN
    B, Kk, H, W = hm.shape
    flat = hm.reshape(B, Kk, -1).argmax(-1)
    iy, ix = (flat // W), (flat % W)
    pad = F.pad(hm, (r, r, r, r), value=float("nan"))
    ar = torch.arange(2 * r + 1, device=hm.device)
    idx_y = (iy[..., None] + ar)[..., None].expand(B, Kk, 2 * r + 1, W + 2 * r)
    rows = torch.gather(pad, 2, idx_y)
    idx_x = (ix[..., None] + ar)[:, :, None, :].expand(B, Kk, 2 * r + 1, 2 * r + 1)
    win = torch.gather(rows, 3, idx_x)
    peaks = torch.stack([ix, iy], -1)
    return peaks.cpu().numpy().astype(np.float64), win.float().cpu().numpy().astype(np.float64)

_GRID = {}
def decode_window(w):
    """w: (2r+1,2r+1) window centred on the integer peak. Returns offsets (dx,dy) per decoder and the fitted covariance."""
    r = w.shape[0] // 2
    out = {"argmax": (0.0, 0.0)}
    peak = w[r, r]
    # quadratic (Taylor step on the log heatmap)
    off = (0.0, 0.0)
    nb = w[r - 1:r + 2, r - 1:r + 2]
    if peak > 0 and np.all(np.isfinite(nb)):
        L = np.log(np.clip(nb, 1e-6 * peak, None))
        gx, gy = (L[1, 2] - L[1, 0]) / 2, (L[2, 1] - L[0, 1]) / 2
        hxx, hyy = L[1, 2] - 2 * L[1, 1] + L[1, 0], L[2, 1] - 2 * L[1, 1] + L[0, 1]
        hxy = (L[2, 2] - L[2, 0] - L[0, 2] + L[0, 0]) / 4
        Hm = np.array([[hxx, hxy], [hxy, hyy]])
        if hxx < 0 and np.linalg.det(Hm) > 0:
            o = -np.linalg.solve(Hm, [gx, gy])
            off = tuple(float(v) for v in np.clip(o, -1, 1))
    out["quadratic"] = off
    # local soft-argmax
    if r not in _GRID:
        g = np.arange(-r, r + 1, dtype=np.float64)
        _GRID[r] = np.meshgrid(g, g)
    gx_, gy_ = _GRID[r]
    pw = np.where(np.isfinite(w), np.clip(w, 0, None), 0.0)
    out["softargmax_local"] = ((pw * gx_).sum() / pw.sum(), (pw * gy_).sum() / pw.sum()) if pw.sum() > 0 else (0.0, 0.0)
    # Gaussian fit (falls back to the quadratic step)
    out["gaussfit"], out["gauss_cov"] = off, (np.nan, np.nan, np.nan)
    m = (np.isfinite(w) & (w > 0.1 * peak)) if peak > 0 else np.zeros(w.shape, bool)
    if m.sum() >= 6:
        X, Y, V = gx_[m], gy_[m], w[m]
        Amat = np.stack([np.ones_like(X), X, Y, X * X, X * Y, Y * Y], 1)
        sw = np.sqrt(V)
        coef, *_ = np.linalg.lstsq(Amat * sw[:, None], np.log(V) * sw, rcond=None)
        A2 = np.array([[coef[3], coef[4] / 2], [coef[4] / 2, coef[5]]])
        if np.all(np.linalg.eigvalsh(A2) < 0):
            S = -0.5 * np.linalg.inv(A2)
            mu = S @ coef[1:3]
            if np.all(np.abs(mu) <= r):
                out["gaussfit"], out["gauss_cov"] = (float(mu[0]), float(mu[1])), (S[0, 0], S[0, 1], S[1, 1])
    return out

## 2. Synthetic images and the run table

In [ ]:
def _render(idx, stream):
    """One image (float32, z-scored), landmark positions (2, 2) [x, y] in px and reference-axis angles (2,) in degrees."""
    rng = np.random.default_rng([stream, idx])
    S = IMG
    yy, xx = np.mgrid[0:S, 0:S].astype(np.float32)
    tex = cv2.GaussianBlur(rng.standard_normal((S, S)).astype(np.float32), (0, 0), max(GEN["texture_sigma_frac"] * S, 0.5))
    img = GEN["texture_amp"] * tex / (tex.std() + 1e-8)
    g = rng.uniform(-0.2, 0.2, 2)
    img += g[0] * (xx / S - 0.5) + g[1] * (yy / S - 0.5)
    lo, hi = GEN["margin_frac"] * S, (1 - GEN["margin_frac"]) * S
    while True:
        p1, p2 = rng.uniform(lo, hi, 2), rng.uniform(lo, hi, 2)
        if np.hypot(*(p1 - p2)) >= GEN["min_landmark_distance_frac"] * S:
            break
    # corner: wedge with apex p1, bisector phi1, opening angle 2*half, arm length L (anti-aliased by signed distance)
    phi1, half = rng.uniform(-math.pi, math.pi), math.radians(rng.uniform(*GEN["wedge_angle_deg"])) / 2
    dx, dy = xx - p1[0], yy - p1[1]
    t = dx * math.cos(phi1) + dy * math.sin(phi1)
    n = -dx * math.sin(phi1) + dy * math.cos(phi1)
    sd_w = np.minimum(t * math.sin(half) - np.abs(n) * math.cos(half), GEN["wedge_len_frac"] * S - t)
    wedge = np.clip(0.5 + sd_w, 0, 1)
    # edge point: region below the curve n = h (exp(-t^2 / 2w^2) - 1) inside a disk around p2; the bump apex is p2
    phi2 = rng.uniform(-math.pi, math.pi)
    h, w = GEN["bump_height_frac"] * S, GEN["bump_width_frac"] * S
    dx, dy = xx - p2[0], yy - p2[1]
    t = dx * math.cos(phi2) + dy * math.sin(phi2)
    n = -dx * math.sin(phi2) + dy * math.cos(phi2)
    nb = h * (np.exp(-t ** 2 / (2 * w ** 2)) - 1)
    slope = -h * t / w ** 2 * np.exp(-t ** 2 / (2 * w ** 2))
    R = GEN["edge_disk_radius_frac"] * S
    off = rng.uniform(-1, 1) * GEN["edge_disk_offset_frac"] * R      # disk centre moved along the edge, so the arc does not mark p2
    sd_e = np.minimum((n - nb) / np.sqrt(1 + slope ** 2), R - np.hypot(t - off, n))
    edge = np.clip(0.5 + sd_e, 0, 1)
    c = rng.uniform(*GEN["contrast"], 2) * rng.choice([-1.0, 1.0], 2)
    img = img + c[0] * wedge + c[1] * edge
    img = cv2.GaussianBlur(img.astype(np.float32), (0, 0), max(GEN["blur_frac"] * S, 0.3))
    img = img + GEN["pixel_noise"] * rng.standard_normal((S, S)).astype(np.float32)
    img = (img - img.mean()) / (img.std() + 1e-8)
    pts = np.array([p1, p2], np.float32)
    ang = np.degrees([phi1, phi2]).astype(np.float32)
    return img, pts, ang

def make_set(n, stream):
    imgs, pts, angs = np.empty((n, IMG, IMG), np.float16), np.empty((n, K, 2), np.float32), np.empty((n, K), np.float32)
    for i in range(n):
        imgs[i], pts[i], angs[i] = _render(i, stream)
    return imgs, pts, angs

def arm_cov(arm, ang_deg):
    """(n, K, 2, 2) target covariances in px^2: round, or equal-area ratio-4 ovals along / across the reference axes."""
    n = ang_deg.shape[0]
    if arm in RW_FACTOR:                       # round-width arm: round (w * sigma0)^2 I
        return np.tile(np.eye(2, dtype=np.float32) * (RW_FACTOR[arm] * SIGMA0_PX) ** 2, (n, K, 1, 1))
    if arm == "B0":
        return np.tile(np.eye(2, dtype=np.float32) * SIGMA0_PX ** 2, (n, K, 1, 1))
    phi = np.radians(ang_deg + (0.0 if arm == "Apar" else 90.0))
    cs, sn = np.cos(phi), np.sin(phi)
    rot = np.stack([np.stack([cs, -sn], -1), np.stack([sn, cs], -1)], -2)
    d = np.zeros(phi.shape + (2, 2), np.float32)
    d[..., 0, 0], d[..., 1, 1] = SIGMA0_PX ** 2 * RATIO, SIGMA0_PX ** 2 / RATIO
    return (rot @ d @ np.swapaxes(rot, -1, -2)).astype(np.float32)

BASE_TABLE = pd.DataFrame([{"run_id": f"C0_SYNTH_{tier}_N{n}_{arm}_s{seed}", "tier": tier, "base": base, "n_train": n, "arm": arm, "seed": seed}
                           for tier, base in TIER_BASE.items() for n in N_TRAIN for seed in SEEDS for arm in ARMS])   # mechanism study
RUN_TABLE = pd.DataFrame([{"run_id": f"C0_SYNTH_{tier}_N{n}_{arm}_s{seed}", "tier": tier, "base": base, "n_train": n, "arm": arm, "seed": seed,
                           "width_factor": RW_FACTOR[arm]}
                          for tier, base in TIER_BASE.items() for n in RW_N for seed in SEEDS for arm in RW_FACTOR])  # this arm
assert RUN_TABLE.run_id.is_unique and not set(RUN_TABLE.run_id) & set(BASE_TABLE.run_id)
PAIRED = BASE_TABLE[BASE_TABLE.n_train.isin(RW_N)]          # B0 / Apar / Aperp runs the analysis pairs with
todo = RUN_TABLE if RUN_IDS == "all" else RUN_TABLE[RUN_TABLE.run_id.isin(RUN_IDS)]
assert RUN_IDS == "all" or len(todo) == len(RUN_IDS), "unknown run ids in RUN_IDS"
N_MAX = int(todo.n_train.max()) if len(todo) else max(RW_N)
t0 = time.time()
TRAIN = make_set(N_MAX, GEN["train_stream"])
TEST = make_set(N_TEST, GEN["test_stream"])
print(f"generated {N_MAX} training and {N_TEST} test images in {time.time() - t0:.1f} s | runs in this notebook: {len(todo)} of {len(RUN_TABLE)}")
RUN_TABLE.to_csv(Path(RUN_ROOT) / "run_table_synth_RW.csv", index=False)

# visual check: two training images with their three targets
SHOW_ARMS = ["B0"] + list(RW_FACTOR)
fig, axs = plt.subplots(2, 1 + len(SHOW_ARMS), figsize=(15, 5.4))
for r in range(2):
    im, pt, an = TRAIN[0][r].astype(np.float32), TRAIN[1][r:r + 1], TRAIN[2][r:r + 1]
    axs[r, 0].imshow(im, cmap="gray"); axs[r, 0].plot(pt[0, :, 0], pt[0, :, 1], "r+", ms=10)
    for j, arm in enumerate(SHOW_ARMS):
        tg = make_targets(torch.tensor(pt), torch.tensor(arm_cov(arm, an)), IMG, IMG)[0].numpy().max(0)
        axs[r, j + 1].imshow(im, cmap="gray"); axs[r, j + 1].imshow(tg, cmap="magma", alpha=0.6)
        axs[r, j + 1].set_title(f"{arm} ({(RW_FACTOR.get(arm, 1.0) * SIGMA0_PX):.2f} px)", fontsize=9)
    for a in axs[r]:
        a.set_xticks([]); a.set_yticks([])
axs[0, 0].set_title("image (+ landmarks)", fontsize=9)
fig.suptitle("Synthetic images: corner and edge point; round targets of the round-width arm (B0 = σ₀ for reference)", fontsize=10)
fig.tight_layout()
fig.savefig(Path(RUN_ROOT) / "synth_RW_examples.png", dpi=120)
RUN_TABLE.groupby(["tier", "n_train"]).size().unstack()   # 8 runs per cell: 4 widths x 2 seeds

## 3. Training and evaluating one run

In [ ]:
try:
    from tqdm.auto import tqdm as _tqdm
except ImportError:
    _tqdm = None

def _grad_scaler():
    enabled = AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def lr_at(it):
    o = SYNTH_PROTOCOL["optimiser"]
    if it < o["warmup_iters"]:
        return o["lr"] * (it + 1) / o["warmup_iters"]
    prog = (it - o["warmup_iters"]) / max(1, ITERS - o["warmup_iters"])
    return 1e-6 + 0.5 * (o["lr"] - 1e-6) * (1 + math.cos(math.pi * prog))

def file_sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

TEST_X = torch.from_numpy(TEST[0])
CONV_N = min(50, N_MAX)
CONV_X = torch.from_numpy(TRAIN[0][:CONV_N])

@torch.no_grad()
def conv_mre(model):
    """MRE (px, quadratic decoder) on the first CONV_N training images: the convergence monitor."""
    model.eval()
    with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
        hm = model(CONV_X.to(DEVICE).float()[:, None])
    peaks, wins = heatmap_windows(hm.float())
    err = []
    for b in range(hm.shape[0]):
        for k in range(K):
            o = decode_window(wins[b, k])["quadratic"]
            err.append(math.hypot(peaks[b, k, 0] + o[0] - TRAIN[1][b, k, 0], peaks[b, k, 1] + o[1] - TRAIN[1][b, k, 1]))
    model.train()
    return float(np.mean(err))
DEC = SYNTH_PROTOCOL["decoders"]

@torch.no_grad()
def predict_test(model):
    model.eval()
    rows = []
    phi = np.radians(TEST[2])
    for s in range(0, N_TEST, 50):
        x = TEST_X[s:s + 50].to(DEVICE).float()[:, None]
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            hm = model(x)
        peaks, wins = heatmap_windows(hm.float())
        for b in range(hm.shape[0]):
            i = s + b
            for k in range(K):
                dec = decode_window(wins[b, k])
                gt = TEST[1][i, k]
                row = {"image": i, "landmark": k, "peak": float(hm[b, k].max())}
                for d in DEC:
                    ex, ey = peaks[b, k, 0] + dec[d][0] - gt[0], peaks[b, k, 1] + dec[d][1] - gt[1]
                    row[f"dx_{d}"], row[f"dy_{d}"] = float(ex), float(ey)
                row["fit_sxx"], row["fit_sxy"], row["fit_syy"] = dec["gauss_cov"]
                ex, ey = row[f"dx_{SYNTH_PROTOCOL['primary_decoder']}"], row[f"dy_{SYNTH_PROTOCOL['primary_decoder']}"]
                row["e_u"] = ex * math.cos(phi[i, k]) + ey * math.sin(phi[i, k])       # along the reference axis
                row["e_v"] = -ex * math.sin(phi[i, k]) + ey * math.cos(phi[i, k])      # across it
                rows.append(row)
    model.train()
    return pd.DataFrame(rows)

def train_run(run):
    out = Path(RUN_ROOT) / run["run_id"]
    if (out / "DONE").exists():
        return "done before"
    out.mkdir(parents=True, exist_ok=True)
    n, seed = int(run["n_train"]), int(run["seed"])
    torch.manual_seed(seed); np.random.seed(seed)
    model = UNet(in_ch=1, out_ch=K, base=int(run["base"])).to(DEVICE).to(memory_format=torch.channels_last)
    o = SYNTH_PROTOCOL["optimiser"]
    opt = torch.optim.AdamW(model.parameters(), lr=o["lr"], weight_decay=o["weight_decay"])
    scaler = _grad_scaler()
    X = torch.from_numpy(TRAIN[0][:n]).to(DEVICE)
    P = torch.from_numpy(TRAIN[1][:n]).to(DEVICE)
    C = torch.from_numpy(arm_cov(run["arm"], TRAIN[2][:n])).to(DEVICE)
    bar = _tqdm(total=ITERS, desc=run["run_id"], leave=True) if PROGRESS_BAR and _tqdm else None
    t_run, log = time.time(), []
    for it in range(ITERS):
        for gparam in opt.param_groups:
            gparam["lr"] = lr_at(it)
        idx = torch.from_numpy(np.random.default_rng([seed, n, it]).integers(0, n, BATCH_SIZE)).to(DEVICE)
        x, p, c = X[idx].float()[:, None].contiguous(memory_format=torch.channels_last), P[idx], C[idx]
        tgt = make_targets(p, c)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            pred = model(x)
        loss = mass_normalised_mse(pred, tgt, c)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if (it + 1) % 100 == 0 or it == ITERS - 1:
            rec = {"iter": it + 1, "loss": float(loss), "lr": lr_at(it), "sec": time.time() - t_run}
            if (it + 1) % max(1, ITERS // 10) == 0 or it == ITERS - 1:
                rec["conv_mre_px"] = conv_mre(model)
            log.append(rec)
            if bar is not None:
                bar.update(it + 1 - bar.n); bar.set_postfix(loss=f"{float(loss):.4f}")
    if bar is not None:
        bar.close()
    pred = predict_test(model)
    pred.to_csv(out / "predictions_test.csv.gz", index=False)
    pd.DataFrame(log).to_csv(out / "train_log.csv", index=False)
    sha = "-"
    if SAVE_MODELS:
        tmp = out / "final.pt"
        torch.save(model.state_dict(), tmp)
        sha = file_sha256(tmp)
        tmp.rename(out / f"final_{sha[-8:]}.pt")
    cfg = {"run": {k: (v.item() if hasattr(v, "item") else v) for k, v in run.items()}, "protocol": SYNTH_PROTOCOL,
           "protocol_hash": SYNTH_HASH, "rw_spec": RW_SPEC, "rw_hash": RW_HASH, "env": ENV_INFO, "model_sha256": sha,
           "train_seconds": time.time() - t_run}
    (out / "config.json").write_text(json.dumps(cfg, indent=1, default=str))
    (out / "DONE").write_text(f"{SYNTH_HASH}\n{sha}\n{RW_HASH}\n")
    return f"trained in {(time.time() - t_run) / 60:.1f} min"

## 4. Execute the selected runs

In [ ]:
for _, run in todo.iterrows():
    print(run.run_id, "->", train_run(run.to_dict()))

## 9. Status of the round-width runs

Training only. The comparison with the mechanism study (B0, A∥, A⊥ of the same tier, N and seed) is in the separate notebook
`C0_SYNTH_round_width_analysis.ipynb`, run locally after downloading the round-width runs. This cell only lists what is done here,
with each finished run's MRE and gross-failure rate as a sanity check (> 5 % failures at a width = the tier cannot learn that blob).

In [ ]:
n_done = sum((Path(RUN_ROOT) / r / "DONE").exists() for r in RUN_TABLE.run_id)
print(f"round-width runs done in RUN_ROOT: {n_done} of {len(RUN_TABLE)}")
stat = []
for _, run in RUN_TABLE.iterrows():
    d = Path(RUN_ROOT) / run.run_id
    if not (d / "DONE").exists():
        continue
    p = pd.read_csv(d / "predictions_test.csv.gz")
    p["r"] = np.hypot(p.e_u, p.e_v)
    for k, g in p.groupby("landmark"):
        stat.append({"tier": run.tier, "n_train": run.n_train, "arm": run.arm, "seed": run.seed, "landmark": LM_NAMES[k],
                     "mre_px": g.r.mean(), "fail_pct": 100 * (g.r > GROSS_PX).mean()})
if stat:
    STAT = pd.DataFrame(stat)
    STAT.round(4).to_csv(Path(RUN_ROOT) / "synth_RW_status.csv", index=False)
    display(STAT.pivot_table(index=["tier", "n_train", "landmark"], columns="arm", values=["mre_px", "fail_pct"]).round(2))